In [20]:
from env import RandomWalkEnv
import numpy as np

In [21]:
env = RandomWalkEnv()

In [28]:
import random

def generate_episode():
    episode = []
    state = env.reset()
    while True:
        action = random.choice(env.action_space)
        next_state, reward, done = env.step(action)
        episode.append((state, action, reward))
        state = next_state
        if done:
            break
    return episode

In [29]:
true_values = np.asarray([0, 1/6, 2/6, 3/6, 4/6, 5/6, 0])
true_values


array([0.        , 0.16666667, 0.33333333, 0.5       , 0.66666667,
       0.83333333, 0.        ])

In [30]:
def constant_alpha_mc_policy_evaluation(env,alpha, num_episodes, gamma=1.0):
    V = np.asarray([0, 0.5, 0.5, 0.5, 0.5, 0.5, 0])   
    for _ in range(num_episodes):
        episode = generate_episode()
        G = 0
        returns = {i:[] for i in range(env.state_space.n)}
        for state,_, reward in reversed(episode):
            G = gamma * G + reward
            returns[state].append(G)
        
        for key,return_value in returns.items():
            for value in return_value:
                V[key] = V[key] + (value - V[key]) * alpha
    return V

alpha_mc_value = constant_alpha_mc_policy_evaluation(env,0.1, 1000)

In [31]:
def td_policy_evaluation(policy, env, alpha, num_episodes, gamma=1.0):
    V = np.asarray([0, 0.5, 0.5, 0.5, 0.5, 0.5, 0])
    
    for _ in range(num_episodes):
        state = env.reset()
        done = False
        while not done:
            action = policy(state)
            next_state, reward, done = env.step(action)
            V[state] += alpha * (reward + gamma * V[next_state] - V[state])
            state = next_state
    return V

td_value = td_policy_evaluation(lambda state: random.choice(env.action_space), env, 0.1, 1000)

In [32]:
def rms_error(values1, values2):
    return np.sqrt(np.mean((values1 - values2) ** 2))

rms_error_td = rms_error(td_value, true_values)
rms_error_alpha_mc = rms_error(alpha_mc_value, true_values)

print(f"RMS Error for TD Value: {rms_error_td}")
print(f"RMS Error for Alpha MC Value: {rms_error_alpha_mc}")

RMS Error for TD Value: 0.06549963679987178
RMS Error for Alpha MC Value: 0.10720156740197949


In [ ]:
import matplotlib.pyplot as plt


mc_alpha_values = [0.01,0.02,0.03]
td_alpha_values = [0.05,0.1,0.15]
num_episodes_values = [i for i in range(1,100,1)]

def simulate_single_run():
    errors = np.zeros((len(mc_alpha_values), len(num_episodes_values),2))
    for i,alpha in enumerate(mc_alpha_values):
        mc_errors = np.zeros(len(num_episodes_values))
        td_errors = np.zeros(len(num_episodes_values))
        for j,num_episodes in enumerate(num_episodes_values):
            values = constant_alpha_mc_policy_evaluation(env,alpha, num_episodes)
            mc_errors[j] = rms_error(values, true_values)
            values = td_policy_evaluation(lambda state: random.choice(env.action_space), env, td_alpha_values[i], num_episodes)
            td_errors[j] = rms_error(values, true_values)
        errors[i,:,0] = mc_errors
        errors[i,:,1] = td_errors
    return errors

number_of_runs = 100

errors = np.zeros((len(mc_alpha_values), len(num_episodes_values),2,number_of_runs))

for i in range(number_of_runs):
    errors[:,:,:,i] = simulate_single_run()

errors = np.mean(errors, axis=3)

for i, alpha in enumerate(mc_alpha_values):
    plt.plot(num_episodes_values, errors[i,:,0], label=f'MC α={alpha}', linestyle='-', marker='o')
    plt.plot(num_episodes_values, errors[i,:,1], label=f'TD α={td_alpha_values[i]}', linestyle='--', marker='x')

plt.xlabel('Number of Episodes')
plt.ylabel('RMS Error')
plt.title('Policy Evaluation Errors for Monte Carlo and TD Methods')
plt.legend()
plt.grid(True)
plt.show()
